# NASA CMAPSS — Data Pipeline
## Combine all 4 datasets into unified train.csv and test.csv

This notebook:
1. Loads all 4 training files (FD001–FD004) and combines them
2. Loads all 4 test files + RUL ground truth and combines them
3. Assigns globally unique `engine_id` across datasets
4. Computes RUL (Remaining Useful Life) for all rows
5. Caps RUL at 125 cycles
6. Saves `data/train.csv` and `data/test.csv`

In [12]:
import pandas as pd
import numpy as np

# Column names: 2 IDs + 3 operational settings + 21 sensors = 26 columns
cols = ['engine_id', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

RUL_CAP = 125  # Cap RUL at 125 cycles (standard practice)

## Load & Combine Training Data

For each FD00X file:
- Load the raw text file
- Add `dataset_id` (1–4) to track which subset it came from
- Offset `engine_id` so IDs are globally unique across all 4 files
- Compute RUL = `max_cycle_for_engine - current_cycle`
- Cap RUL at 125

In [13]:
train_dfs = []
engine_id_offset = 0

for i in range(1, 5):
    df = pd.read_csv(f'data/train_FD00{i}.txt', sep=r'\s+', header=None, names=cols)
    df['dataset_id'] = i

    # Make engine_id globally unique across all 4 datasets
    df['engine_id'] = df['engine_id'] + engine_id_offset
    engine_id_offset = df['engine_id'].max()

    # Compute RUL: for training data, last cycle = failure
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    df = df.merge(max_cycles, on='engine_id')
    df['RUL'] = df['max_cycle'] - df['cycle']
    df.drop('max_cycle', axis=1, inplace=True)

    train_dfs.append(df)
    print(f"FD00{i}: {df['engine_id'].nunique()} engines, {len(df)} rows, "
          f"engine_id range: {df['engine_id'].min()}-{df['engine_id'].max()}")

train = pd.concat(train_dfs, ignore_index=True)

# Cap RUL at 125
train['RUL'] = train['RUL'].clip(upper=RUL_CAP)

print(f"\n--- Combined Training Data ---")
print(f"Shape: {train.shape}")
print(f"Total engines: {train['engine_id'].nunique()}")
print(f"RUL range: {train['RUL'].min()} to {train['RUL'].max()}")
train.head()

FD001: 100 engines, 20631 rows, engine_id range: 1-100
FD002: 260 engines, 53759 rows, engine_id range: 101-360
FD003: 100 engines, 24720 rows, engine_id range: 361-460
FD004: 249 engines, 61249 rows, engine_id range: 461-709

--- Combined Training Data ---
Shape: (160359, 28)
Total engines: 709
RUL range: 0 to 125


,engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,dataset_id,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,1,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,1,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,1,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,1,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,1,125


## Load & Combine Test Data + RUL Ground Truth

For test data, the time series is truncated before failure. The `RUL_FD00X.txt` file provides the true RUL at each engine's **last observed cycle**.

We compute RUL for every row in the test set:
- `RUL_at_row = RUL_at_last_cycle + (max_cycle - current_cycle)`

In [14]:
test_dfs = []
engine_id_offset = 0

for i in range(1, 5):
    df = pd.read_csv(f'data/test_FD00{i}.txt', sep=r'\s+', header=None, names=cols)
    rul = pd.read_csv(f'data/RUL_FD00{i}.txt', header=None, names=['RUL'])

    df['dataset_id'] = i

    # Make engine_id globally unique
    df['engine_id'] = df['engine_id'] + engine_id_offset
    engine_id_offset = df['engine_id'].max()

    # Map RUL to each engine (RUL file has one value per engine, in order)
    unique_engines = sorted(df['engine_id'].unique())
    rul_map = pd.DataFrame({'engine_id': unique_engines, 'rul_last': rul['RUL'].values})

    # Compute RUL for every row:
    # RUL_at_row = RUL_at_last_cycle + (last_cycle - current_cycle)
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    df = df.merge(max_cycles, on='engine_id')
    df = df.merge(rul_map, on='engine_id')
    df['RUL'] = df['rul_last'] + (df['max_cycle'] - df['cycle'])
    df.drop(['max_cycle', 'rul_last'], axis=1, inplace=True)

    test_dfs.append(df)
    print(f"FD00{i}: {df['engine_id'].nunique()} engines, {len(df)} rows, "
          f"engine_id range: {df['engine_id'].min()}-{df['engine_id'].max()}")

test = pd.concat(test_dfs, ignore_index=True)

# Cap RUL at 125 for consistency
test['RUL'] = test['RUL'].clip(upper=RUL_CAP)

print(f"\n--- Combined Test Data ---")
print(f"Shape: {test.shape}")
print(f"Total engines: {test['engine_id'].nunique()}")
print(f"RUL range: {test['RUL'].min()} to {test['RUL'].max()}")
test.head()

FD001: 100 engines, 13096 rows, engine_id range: 1-100
FD002: 259 engines, 33991 rows, engine_id range: 101-359
FD003: 100 engines, 16596 rows, engine_id range: 360-459
FD004: 248 engines, 41214 rows, engine_id range: 460-707

--- Combined Test Data ---
Shape: (104897, 28)
Total engines: 707
RUL range: 6 to 125


,engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,dataset_id,RUL
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735,1,125
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916,1,125
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166,1,125
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737,1,125
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130,1,125


## Save to CSV

In [15]:
train.to_csv('data/train.csv', index=False)
test.to_csv('data/test.csv', index=False)
print("Saved: data/train.csv and data/test.csv")

Saved: data/train.csv and data/test.csv


## Verification

In [16]:
print("=== VERIFICATION ===\n")

# Check engine counts
print(f"Train engines: {train['engine_id'].nunique()} (expected: 100+260+100+248 = 708)")
print(f"Test engines:  {test['engine_id'].nunique()} (expected: 100+259+100+249 = 708)")

# Check RUL range
print(f"\nTrain RUL range: {train['RUL'].min()} to {train['RUL'].max()} (should be 0 to {RUL_CAP})")
print(f"Test RUL range:  {test['RUL'].min()} to {test['RUL'].max()}")

# Check that last row of each training engine has RUL = 0
last_rows = train.groupby('engine_id')['RUL'].last()
assert (last_rows == 0).all(), "ERROR: Not all training engines end with RUL=0!"
print("\nAll training engines end with RUL=0")

# Check dataset distribution
print(f"\nDataset distribution (train):")
print(train.groupby('dataset_id')['engine_id'].nunique().to_string())

print(f"\nDataset distribution (test):")
print(test.groupby('dataset_id')['engine_id'].nunique().to_string())

# Column check
print(f"\nColumns: {list(train.columns)}")
print(f"\nTrain shape: {train.shape}")
print(f"Test shape:  {test.shape}")

=== VERIFICATION ===

Train engines: 709 (expected: 100+260+100+248 = 708)
Test engines:  707 (expected: 100+259+100+249 = 708)

Train RUL range: 0 to 125 (should be 0 to 125)
Test RUL range:  6 to 125

All training engines end with RUL=0

Dataset distribution (train):
dataset_id
1    100
2    260
3    100
4    249

Dataset distribution (test):
dataset_id
1    100
2    259
3    100
4    248

Columns: ['engine_id', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'dataset_id', 'RUL']

Train shape: (160359, 28)
Test shape:  (104897, 28)
